# MAE training on Runpod

Single-GPU MAE on STL-10 unlabeled. Linear-scaled LR from `base_lr`, native bf16 (no autocast / GradScaler), full checkpoints to Hugging Face every 20 epochs, loss logged to wandb project `mae`.

Expects Runpod secrets `HF_TOKEN` and `WANDB_API_KEY`.


In [ ]:
import json
import logging
import os
import sys

import torch
import torchvision
import torchvision.transforms as transforms
import wandb
from huggingface_hub import HfApi, login as hf_login, utils as hf_utils
from torch import nn, optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader

from model import Net, get_pos_embeddings

hf_utils.disable_progress_bars()

# Logger for Jupyter notebook
logger = logging.getLogger("mae_train")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_h = logging.StreamHandler(sys.stdout)
_h.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%H:%M:%S"))
logger.addHandler(_h)
logger.propagate = False


In [1]:
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True

with open(os.path.join("checkpoints", "new_config.json"), "r") as f:
    cfg = json.load(f)

# --- Base params ---
D_IMAGE    = cfg["data"]["d_image"]
N_CHANNELS = cfg["data"]["n_channels"]
PATCH_SIZE = cfg["data"]["patch_size"]

D_ENC            = cfg["model"]["d_enc"]
N_HEADS_ENC      = cfg["model"]["n_heads_enc"]
N_ENCODER_BLOCKS = cfg["model"]["n_encoder_blocks"]
MLP_RATIO        = cfg["model"]["mlp_ratio"]

D_DEC            = cfg["model"]["d_dec"]
N_HEADS_DEC      = cfg["model"]["n_heads_dec"]
N_DECODER_BLOCKS = cfg["model"]["n_decoder_blocks"]

DATA_PATH        = cfg["metadata"]["data_path"]
BASE_LR          = cfg["metadata"]["base_lr"]
BATCH_SIZE       = cfg["metadata"]["batch_size"]
WEIGHT_DECAY     = cfg["metadata"]["weight_decay"]
MOMENTUM         = cfg["metadata"]["momentum"]
WARMUP_EPOCHS    = cfg["metadata"]["warmup_epochs"]
N_EPOCHS         = cfg["metadata"]["n_epochs"]
PERCENT_UNMASKED = cfg["metadata"]["percent_unmasked"]
CHECKPOINT_EVERY = cfg["metadata"]["checkpoint_every"]
OUTPUT_DIR       = cfg["metadata"]["output_dir"]
CHECKPOINT_DIR   = cfg["metadata"]["checkpoint_dir"]
HF_REPO          = cfg["metadata"]["hf_repo"]
HF_OUTPUT_DIR    = cfg["metadata"]["hf_output_dir"]
WANDB_PROJECT    = cfg["metadata"]["wandb_project"]

# MAE linear LR scaling: lr = base_lr * batch_size / 256
LR = BASE_LR * BATCH_SIZE / 256
cfg["metadata"]["lr"] = LR

D_PATCH   = (PATCH_SIZE ** 2) * N_CHANNELS
N_PATCHES = (D_IMAGE ** 2) // (PATCH_SIZE ** 2)
N_ROWS    = D_IMAGE // PATCH_SIZE
D_ENC_MLP = int(MLP_RATIO * D_ENC)
D_DEC_MLP = int(MLP_RATIO * D_DEC)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
logger.info(f"base_lr={BASE_LR}  batch_size={BATCH_SIZE}  lr={LR}")


NameError: name 'torch' is not defined

In [ ]:
hf_login(token=os.environ["HF_TOKEN"])
hf_api = HfApi()

config_path = os.path.join(OUTPUT_DIR, "config.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

hf_api.upload_file(
    path_or_fileobj=config_path,
    path_in_repo=os.path.join(HF_OUTPUT_DIR, "config.json"),
    repo_id=HF_REPO,
    repo_type="model",
)

def upload_file_hf(path):
    hf_api.upload_file(
        path_or_fileobj=path,
        path_in_repo=os.path.join(HF_OUTPUT_DIR, os.path.basename(path)),
        repo_id=HF_REPO,
        repo_type="model",
    )

wandb.login(key=os.environ["WANDB_API_KEY"])
wandb.init(project=WANDB_PROJECT, config=cfg, name=HF_OUTPUT_DIR)
logger.info(f"HF repo={HF_REPO}/{HF_OUTPUT_DIR}  wandb project={WANDB_PROJECT}")


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(D_IMAGE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4467, 0.4398, 0.4066), (0.2604, 0.2566, 0.2713)),
])

unlabeled_set = torchvision.datasets.STL10(
    root=DATA_PATH, split="unlabeled", download=True, transform=train_transform
)
trainloader = DataLoader(
    unlabeled_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    drop_last=True,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)
logger.info(f"STL-10 unlabeled: {len(unlabeled_set)} images, {len(trainloader)} steps/epoch")


In [ ]:
pos_embeddings_enc = get_pos_embeddings(D_ENC, N_PATCHES, N_ROWS)
pos_embeddings_dec = get_pos_embeddings(D_DEC, N_PATCHES, N_ROWS)

model = Net(
    n_encoder_blocks=N_ENCODER_BLOCKS,
    n_decoder_blocks=N_DECODER_BLOCKS,
    d_image=D_IMAGE,
    patch_size=PATCH_SIZE,
    d_patch=D_PATCH,
    n_patches=N_PATCHES,
    n_rows=N_ROWS,
    d_enc=D_ENC,
    d_enc_mlp=D_ENC_MLP,
    d_dec=D_DEC,
    d_dec_mlp=D_DEC_MLP,
    n_heads_enc=N_HEADS_ENC,
    n_heads_dec=N_HEADS_DEC,
    pos_embeddings_enc=pos_embeddings_enc,
    pos_embeddings_dec=pos_embeddings_dec,
    percent_unmasked=PERCENT_UNMASKED,
)
model.to(device=device, dtype=torch.bfloat16)

criterion = nn.MSELoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=tuple(MOMENTUM),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)
warmup = LinearLR(optimizer, start_factor=1 / WARMUP_EPOCHS, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(optimizer, N_EPOCHS - WARMUP_EPOCHS)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[WARMUP_EPOCHS])


In [ ]:
def save_checkpoint(state, path):
    torch.save(state, path)
    upload_file_hf(path)
    return path

try:
    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for data in trainloader:
            inputs = data[0].to(device, dtype=torch.bfloat16, non_blocking=True)

            y_pred_patches, y_true_patches, ids_masked = model(inputs)
            masked_indices = ids_masked.unsqueeze(-1).expand(-1, -1, D_PATCH)
            y_pred_masked = torch.gather(y_pred_patches, 1, masked_indices)

            with torch.no_grad():
                y_true_masked = torch.gather(y_true_patches, 1, masked_indices)
                mean = y_true_masked.mean(dim=-1, keepdim=True)
                var = y_true_masked.var(dim=-1, keepdim=True)
                y_true_masked = (y_true_masked - mean) / (var + 1e-6).sqrt()

            loss = criterion(y_pred_masked, y_true_masked)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(trainloader)
        current_lr = optimizer.param_groups[0]["lr"]
        wandb.log({"loss": avg_loss, "lr": current_lr}, step=epoch + 1)
        logger.info(f"Epoch {epoch + 1}/{N_EPOCHS} - loss: {avg_loss:.6f}  lr: {current_lr:.6e}")

        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            path = save_checkpoint(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "loss": avg_loss,
                },
                os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch + 1}.pt"),
            )
            logger.info(f"Checkpoint saved: {path}")
finally:
    wandb.finish()
    logger.info("Finished training")
